In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt 
from matplotlib.lines import Line2D

In [ ]:
delta_A = np.array([[np.sqrt(3), -np.sqrt(3), 0], [1, 1, -2]])/2
eta_odd = np.array([[2*np.sqrt(3), -np.sqrt(3), -np.sqrt(3)], [0, 3, -3]])/2
Kp = np.array([4*np.pi/(3*np.sqrt(3)), 0])

phi = np.arctan(1)
vs, vsp, vd = 1/2, 0.1/2, 0.1/2
v0, vt = (3*vs + 6*vsp + 0.1), np.sqrt(vsp**2 + vd**2)

def rho_p_k(kx, ky):
    rhok = np.zeros(kx.shape)
    pk = np.zeros(kx.shape)
    for i in range(3):
        x = eta_odd[0][i] * kx
        y = eta_odd[1][i] * ky
        rhok += np.sin(x+y)
        pk += np.cos(x+y)
    return rhok, pk

def h_i(kx, ky):
    rhok, pk = rho_p_k(kx, ky)
    
    h0 = v0*np.ones(kx.shape)-2*vt*np.cos(phi)*pk
    hz = 2*vt*np.sin(phi)*rhok
    hx, hy = np.zeros(kx.shape), np.zeros(kx.shape)
    for i in range(3):
        x = delta_A[0][i] * kx
        y = delta_A[1][i] * ky
        hx += -vs*np.cos(x+y)
        hy += -vs*np.sin(x+y)
    return h0, hx, hy, hz

def e_minus(kx, ky):
    h0, hx, hy, hz = h_i(kx, ky)
    return h0 - np.sqrt(hx**2 + hy**2 + hz**2)

def Phi(kx, ky):
    h0, hx, hy, hz = h_i(kx, ky)
    phase = (hx-1j*hy)/np.sqrt(hx**2+hy**2) * np.exp(1j*np.pi)
    Phik = 1j*np.log(phase)
    return Phik.real

def partial_Phi(kx, ky, d, axis=1):
    """
    Numerical d(Phi)/d(kx) via finite differences along `axis`
    (default axis=1, i.e. kx varies along columns as in a standard
    np.meshgrid(kx_1d, ky_1d) with indexing='xy').

    Edge points: one-sided difference
        dPhi[0]  = Phi[1]  - Phi[0]
        dPhi[-1] = Phi[-1] - Phi[-2]
    Interior points: central difference
        dPhi[i]  = Phi[i+1] - Phi[i-1]
        
    dx: axis=1
    dy: axis=0
    """
    Phik = Phi(kx, ky)

    # bring the derivative axis to the end for easy slicing
    Phik_m = np.moveaxis(Phik, axis, -1)
    dPhi_m = np.empty_like(Phik_m)

    dPhi_m[..., 1:-1] = (Phik_m[..., 2:] - Phik_m[..., :-2])/(2*d)   # central
    dPhi_m[..., 0]    = (Phik_m[..., 1] - Phik_m[..., 0])/d      # forward
    dPhi_m[..., -1]   = (Phik_m[..., -1] - Phik_m[..., -2])/d    # backward

    return np.moveaxis(dPhi_m, -1, axis)

def partial_hz(kx, ky, d, axis=1):
    """
    dx: axis=1
    dy: axis=0
    """
    _, _, _, hz = h_i(kx, ky)
    ep = e_minus(kx, ky)
    func = hz/ep

    func_m = np.moveaxis(func, axis, -1)
    dfunc_m = np.empty_like(func_m)

    dfunc_m[..., 1:-1] = (func_m[..., 2:] - func_m[..., :-2])/(2*d)   # central
    dfunc_m[..., 0]    = (func_m[..., 1] - func_m[..., 0])/d      # forward
    dfunc_m[..., -1]   = (func_m[..., -1] - func_m[..., -2])/d    # backward

    return np.moveaxis(dfunc_m, -1, axis)

def omega(kx, ky, dx, dy):
    
    A = partial_Phi(kx, ky, dx, axis=1)*partial_hz(kx, ky, dy, axis=0)
    B = partial_Phi(kx, ky, dy, axis=0)*partial_hz(kx, ky, dx, axis=1)
    return -1/2 * (A-B)
    

def n_b(kx, ky, T):
    if T<0.1: 
        return np.exp(-e_minus(kx, ky)/T)
    else:
        return 1/(np.exp(e_minus(kx, ky)/T)-np.ones(kx.shape))
    

In [ ]:
fig, ax = plt.subplots()

x_end, y_end = 2*np.pi/(3*np.sqrt(3)), 2*np.pi/(3*np.sqrt(3))
kx, dx = np.linspace(0, 1*x_end, num=101, retstep=True)
ky, dy = np.linspace(0, 1*y_end, num=101, retstep=True)
xx, yy = np.meshgrid(kx, ky)

# mesh = ax.pcolormesh(xx, yy, e_minus(xx, yy))
# mesh = ax.pcolormesh(xx, yy, Phi(xx, yy))
mesh = ax.pcolormesh(xx, yy, omega(xx, yy, dx, dy))
# mesh = ax.pcolormesh(xx, yy, partial_hz(xx, yy, dy, axis=0))
cbar = fig.colorbar(mesh, ax=ax, pad=0.01)

plt.show()

# $n_-$

In [ ]:
delta_A = np.array([[np.sqrt(3), 1], [-np.sqrt(3), 1], [0, -2]])/2
eta_odd = np.array([[2*np.sqrt(3), 0], [-np.sqrt(3), 3], [-np.sqrt(3), -3]])/2
Kp = np.array([4*np.pi/(3*np.sqrt(3)), 0])

phi = np.arctan(1)
vs, vsp, vd = 1/2, 0.1/2, 0.1/2
v0, vt = (3*vs + 6*vsp + 0.1), np.sqrt(vsp**2 + vd**2)

def rho_p_k(k):
    rhok = 0
    pk = 0
    for i in range(3):
        x = (eta_odd[i] * k).sum()
        rhok += np.sin(x)
        pk += np.cos(x)
    return rhok, pk

def h_i(k):
    rhok, pk = rho_p_k(k)
    h0 = v0-2*vt*np.cos(phi)*pk
    hz = 2*vt*np.sin(phi)*rhok
    hx, hy = 0, 0
    for i in range(3):
        x = (delta_A[i] * k).sum()
        hx += -vs*np.cos(x)
        hy += -vs*np.sin(x)
    return h0, hx, hy, hz

def e_minus(k):
    h0, hx, hy, hz = h_i(k)
    return h0 - np.sqrt(hx**2 + hy**2 + hz**2)

def Phi(k):
    h0, hx, hy, hz = h_i(k)
    phase = (hx-1j*hy)/np.sqrt(hx**2+hy**2)
    Phik = 1j*np.log(phase)
    return Phik

def n_b(k, T):
    if T<0.005: # e_minus>0.1, e_minus/T > 20
        return 0
    else:
        return 1/(np.exp(e_minus(k)/T)-1)

In [ ]:
from matplotlib.lines import Line2D

fig, ax = plt.subplots(dpi=150)

momenta = np.linspace(0, 1, 100)

for m in momenta:
    ax.scatter(m, e_minus(m*Kp), c='C0', marker='.')
    ax.scatter(m, n_b(m*Kp, 0.1), c='C2', marker='^')
    ax.scatter(m, n_b(m*Kp, 0.2), c='C3', marker='^')
    

legend_elements = [
    Line2D([0], [0], c='C0',
        markersize=5, marker='.', linestyle='', label='$\\epsilon_-$'),
    Line2D([0], [0], c='C2',
        markersize=5, marker='^', linestyle='', label='$n_-(T=0.1)$'),
    Line2D([0], [0], c='C3',
        markersize=5, marker='^', linestyle='', label='$n_-(T=0.2)$'),
    ]
ax.legend(handles=legend_elements, fontsize=15)
ax.set_xlabel('$K^-$')
ax.set_ylabel('$\\omega$')
    
plt.show()